# 11 Active Learning: 

In active learning we want:

* To define a series of pulses on the conditional signal and signal
* Evaluate the response based on the presence of both elements
* Check that after exposure to the pattern learning happens (CS -> O only if in training CS correlated with S)
* Check that the "only if" is valid by sending under a random pattern where CS and S are uncorrelated, there is no or minimal response.

## Step 1: imports

We use `CVODE` here to match the other RL4CRN tutorial notebooks.


In [ ]:
from typing import Any, Callable, Dict, List

import matplotlib.pyplot as plt
import numpy as np

from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.crn_builders import build_simple_IOCRN
from RL4CRN.utils.input_interface import (
    SolverCfg,
    TaskKindBase,
    TaskSpec,
    make_task,
    register_task_kind,
    _TASK_KIND_REGISTRY,
)
from RL4CRN.utils.library_builders import build_MAK_library


## Step 2: build a tiny CRN template

Classic IO-CRN with 4 species, 1 output and 2 input + 1 internal node. We don't allow production other than the ones expressed in the template


In [ ]:
# configure the deterministic solver used by every candidate simulation
solver = SolverCfg(algorithm="CVODE", rtol=1e-8, atol=1e-10)

# build the smallest template that can receive a pulse, remember it in X, and report through Y
crn_template, species_labels = build_simple_IOCRN(
    species=["X1", "X2", "H", "O"],
    production_input_map={"X1": "cs", "X2": "s"},
    output_species="O",
    dilution_map={"H": 0.1},
    solver=solver,
)

# build the first-order mass-action reaction library available to generated candidates
library_components = build_MAK_library(
    crn_template=crn_template,
    species_labels=species_labels,
    order=2,
)
library, M, K, masks = library_components

# select spontaneous production reactions: they have no reactants and are not part of the template
ids_to_remove = [
    reaction.ID
    for reaction in library.reactions
    if len(reaction.reactant_labels) == 0
    and len(reaction.product_labels) > 0
    and reaction not in crn_template.reactions
]

# remove those reactions through the library API; fixed template reactions are intentionally separate
library.remove_reactions(ids_to_remove, remove_by="ID")

# crucially, update the template's library context to reflect the new library state
crn_template.set_library_context(library)

# rebuild the tuple because removing reactions changes its sizes and masks (take as is)
library_components = (
    library,
    len(library.reactions),
    library.get_num_parameters(),
    {
        "continuous": library.get_parameter_mask(mode="continuous", force=True),
        "discrete": library.get_parameter_mask(mode="discrete", force=True),
        "logit": library.get_logit_mask(force=True),
    },
)

print(library)
print(f"Removed {len(ids_to_remove)} spontaneous production reactions from the library.")


## Step 3: define the custom TaskKind


In [ ]:
import numpy as np


def sample_interval_events(
    n,
    time_sampler,
    length_sampler,
    value,
    *,
    rng=None,
    min_length=0.0,
):
    """
    Sample intervals and return piecewise-constant events.

    Convention:
        [(t, v), ...] means: at time t, value becomes v.
        The implicit value at time 0 is 0.

    Each sampled interval [start, start + length) turns `value` on,
    then turns it back to 0. Overlapping intervals are merged.
    """
    rng = np.random.default_rng(rng)

    starts = np.asarray(time_sampler(rng, n), dtype=float)
    lengths = np.asarray(length_sampler(rng, n), dtype=float)
    lengths = np.maximum(lengths, min_length)

    intervals = sorted(
        (max(0.0, s), s + l)
        for s, l in zip(starts, lengths)
        if l > 0 and s + l > 0
    )

    merged = []
    for start, end in intervals:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    events = []
    for start, end in merged:
        if start > 0:
            events.append((start, value))
        else:
            events.append((0.0, value))

        events.append((end, 0.0))

    return events


def combine_piecewise_events(seq_a, seq_b, default_a=0.0, default_b=0.0):
    """
    Combine two event-style piecewise-constant sequences.

    Inputs:
        seq_a = [(time, value_a), ...]
        seq_b = [(time, value_b), ...]

    Convention:
        At each time, the sequence value becomes the provided value.
        Before the first event, the value is default_a/default_b.

    Returns:
        [(time, (value_a, value_b)), ...]
    """
    events = []

    for t, v in seq_a:
        events.append((float(t), "a", v))

    for t, v in seq_b:
        events.append((float(t), "b", v))

    events.sort(key=lambda x: x[0])

    out = []
    val_a = default_a
    val_b = default_b

    i = 0
    while i < len(events):
        t = events[i][0]

        while i < len(events) and np.isclose(events[i][0], t):
            _, which, v = events[i]

            if which == "a":
                val_a = v
            else:
                val_b = v

            i += 1

        pair = (val_a, val_b)

        if out and out[-1][1] == pair:
            out[-1] = (t, pair)
        else:
            out.append((t, pair))

    return out


# -------------------------
# Test / example
# -------------------------

def uniform_times(rng, n):
    return rng.uniform(0, 10, size=n)


def exponential_lengths(rng, n):
    return rng.exponential(scale=2.0, size=n)


def normal_lengths(rng, n):
    return np.abs(rng.normal(loc=1.5, scale=0.5, size=n))


seq_a = sample_interval_events(
    n=8,
    time_sampler=uniform_times,
    length_sampler=exponential_lengths,
    value=1.0,
    rng=1,
)

seq_b = sample_interval_events(
    n=8,
    time_sampler=uniform_times,
    length_sampler=normal_lengths,
    value=5.0,
    rng=2,
)

combined = combine_piecewise_events(seq_a, seq_b)

print("seq_a:", seq_a)
print("seq_b:", seq_b)
print("combined:", combined)


# Simple deterministic check
a = [(0, 1), (2, 0), (5, 1), (7, 0)]
b = [(1, 10), (3, 0), (6, 20), (8, 0)]

expected = [
    (0, (1, 0.0)),
    (1, (1, 10)),
    (2, (0, 10)),
    (3, (0, 0)),
    (5, (1, 0)),
    (6, (1, 20)),
    (7, (0, 20)),
    (8, (0, 0)),
]

got = combine_piecewise_events(a, b)
assert got == expected, got

print("Test passed.")

In [ ]:
# make this definition cell re-runnable because the registry rejects duplicate task names
_TASK_KIND_REGISTRY.pop("active_learning", None)


def _sampler_from_spec(spec):
    """Turn a callable, constant, or small string expression into sampler(rng, n)."""
    if callable(spec):
        return spec

    if isinstance(spec, (int, float)):
        value = float(spec)
        return lambda rng, n: np.full(n, value, dtype=float)

    if isinstance(spec, str):
        allowed = {"np": np}

        def sampler(rng, n):
            values = []
            for _ in range(n):
                allowed["rng"] = rng
                values.append(float(eval(spec, {"__builtins__": {}}, allowed)))
            return np.asarray(values, dtype=float)

        return sampler

    raise TypeError(f"Unsupported sampler spec: {spec!r}")


def _events_to_piecewise_protocol(events, end_time, n_t_per_interval, n_inputs=2):
    """
    Convert event pairs [(t, values), ...] into RL4CRN piecewise inputs.

    Event convention: at time t, the value becomes values. The implicit value at
    t=0 is all zeros. RL4CRN expects segment durations plus one input vector per
    segment, so this function converts absolute event times into that format.
    """
    current = np.zeros(n_inputs, dtype=np.float32)
    breakpoints = [0.0]
    values = []

    for t, event_value in sorted(events, key=lambda item: item[0]):
        t = float(t)
        if t < 0.0 or t > end_time:
            continue

        if t > breakpoints[-1]:
            values.append(current.copy())
            breakpoints.append(t)

        current = np.asarray(event_value, dtype=np.float32).reshape(n_inputs)

    if end_time > breakpoints[-1]:
        values.append(current.copy())
        breakpoints.append(float(end_time))

    u_sequence = []
    nested_time_horizon = []
    for start, end, value in zip(breakpoints[:-1], breakpoints[1:], values):
        duration = float(end - start)
        if duration <= 0.0:
            continue
        u_sequence.append(value)
        nested_time_horizon.append(np.linspace(0.0, duration, n_t_per_interval, dtype=np.float32))

    if not u_sequence:
        u_sequence = [current.copy()]
        nested_time_horizon = [np.linspace(0.0, float(end_time), n_t_per_interval, dtype=np.float32)]

    return u_sequence, nested_time_horizon


def _sample_active_learning_protocol(
    rng,
    *,
    n_peaks,
    horizon,
    amplitude,
    start_sampler,
    length_sampler,
    delay_sampler,
    correlated,
    n_t_per_interval,
):
    cs_events = sample_interval_events(
        n=n_peaks,
        time_sampler=start_sampler,
        length_sampler=length_sampler,
        value=amplitude,
        rng=rng,
        min_length=1e-6,
    )

    if correlated:
        cs_on_events = [(t, v) for t, v in cs_events if v != 0.0]
        starts = np.asarray([t for t, _ in cs_on_events], dtype=float)
        delays = np.asarray(delay_sampler(rng, len(starts)), dtype=float)
        lengths = np.asarray(length_sampler(rng, len(starts)), dtype=float)
        s_starts = np.clip(starts + np.maximum(delays, 0.0), 0.0, horizon)
        s_lengths = np.maximum(lengths, 1e-6)

        intervals = sorted(
            (float(start), float(min(start + length, horizon)))
            for start, length in zip(s_starts, s_lengths)
            if start < horizon and length > 0.0
        )
        s_events = []
        for start, end in intervals:
            if not s_events or start > s_events[-1][0]:
                s_events.append((start, amplitude))
                s_events.append((end, 0.0))
            else:
                s_events[-1] = (max(s_events[-1][0], end), 0.0)
    else:
        s_events = sample_interval_events(
            n=n_peaks,
            time_sampler=start_sampler,
            length_sampler=length_sampler,
            value=amplitude,
            rng=rng,
            min_length=1e-6,
        )

    combined_events = combine_piecewise_events(cs_events, s_events, default_a=0.0, default_b=0.0)
    u_sequence, nested_time_horizon = _events_to_piecewise_protocol(
        combined_events,
        end_time=horizon,
        n_t_per_interval=n_t_per_interval,
        n_inputs=2,
    )
    return combined_events, u_sequence, nested_time_horizon


def _target_trace_from_protocol(u_sequence, nested_time_horizon, *, input_index):
    """Build a target trace aligned to transient_response_piecewise output time points."""
    pieces = []
    for u_step, t_segment in zip(u_sequence, nested_time_horizon):
        value = float(np.asarray(u_step, dtype=float).reshape(-1)[input_index])
        pieces.append(np.full(len(t_segment), value, dtype=float))
    return np.concatenate(pieces) if pieces else np.empty((0,), dtype=float)


# register the class so make_task(...) can construct it through the common task interface
@register_task_kind
class ActiveLearningTaskKind(TaskKindBase):
    """
    Active-learning task with two input channels: CS and S.

    Correlated protocols should make output O follow the S input trace;
    uncorrelated protocols should keep O near zero. Inputs are sampled as
    event-style interval sequences, merged, then converted to RL4CRN
    piecewise-constant protocols.
    """

    kind = "active_learning"

    @staticmethod
    def help() -> Dict[str, Any]:
        return {
            "required": {
                "start_distribution": "Sampler for interval start times, e.g. lambda rng, n: rng.uniform(0, 20, n).",
                "duration_distribution": "Sampler for interval lengths, e.g. lambda rng, n: rng.exponential(1.0, n).",
                "correlation_delay_distribution": "Sampler for S delay after CS in correlated protocols.",
                "horizon": "Total simulated protocol duration.",
                "n_samples": "Number of correlated and uncorrelated protocols to sample.",
                "n_peaks": "Number of CS/S intervals per sampled protocol.",
            },
            "optional": {
                "pulse_amplitude": "Input amplitude during each interval. Default: 1.0.",
                "on_weight": "Weight for correlated O-vs-S trajectory error. Default: 1.0.",
                "off_weight": "Weight for uncorrelated-protocol error. Default: 1.0.",
                "n_t_per_interval": "Simulation points per constant-input interval. Default: 25.",
                "seed": "Optional RNG seed for reproducible protocol sampling.",
                "ic": "Initial condition specification. Default: zero.",
            },
            "notes": "Lower loss is better. Correlated protocols target O(t)=S(t); uncorrelated protocols target O(t)=0.",
        }

    def validate(self, task: TaskSpec) -> None:
        required = (
            "start_distribution",
            "duration_distribution",
            "correlation_delay_distribution",
            "horizon",
            "n_samples",
            "n_peaks",
        )
        for key in required:
            if key not in task.params:
                raise ValueError(f"active_learning task requires params[{key!r}].")

        if task.n_inputs != 2:
            raise ValueError("active_learning expects exactly two input channels: CS and S.")
        if len(task.template_crn.output_labels) != 1:
            raise ValueError("active_learning expects exactly one output species.")
        if float(task.params.get("pulse_amplitude", 1.0)) <= 0:
            raise ValueError("pulse_amplitude must be positive.")
        if float(task.params["horizon"]) <= 0:
            raise ValueError("horizon must be positive.")
        if int(task.params["n_samples"]) < 1:
            raise ValueError("n_samples must be at least 1.")
        if int(task.params["n_peaks"]) < 1:
            raise ValueError("n_peaks must be at least 1.")
        if int(task.params.get("n_t_per_interval", 25)) < 2:
            raise ValueError("n_t_per_interval must be at least 2.")
        if float(task.params.get("on_weight", 1.0)) <= 0:
            raise ValueError("on_weight must be positive.")
        if float(task.params.get("off_weight", 1.0)) <= 0:
            raise ValueError("off_weight must be positive.")

    def default_u_list(self, task: TaskSpec) -> List[np.ndarray]:
        pulse_amplitude = float(task.params.get("pulse_amplitude", 1.0))
        return [np.array([pulse_amplitude, pulse_amplitude], dtype=np.float32)]

    def make_reward_fn(self, task: TaskSpec, overrides: Dict[str, Any]) -> Callable[[Any], Any]:
        pulse_amplitude = float(overrides.get("pulse_amplitude", task.params.get("pulse_amplitude", 1.0)))
        on_weight = float(overrides.get("on_weight", task.params.get("on_weight", 1.0)))
        off_weight = float(overrides.get("off_weight", task.params.get("off_weight", 1.0)))
        horizon = float(overrides.get("horizon", task.params["horizon"]))
        n_samples = int(overrides.get("n_samples", task.params["n_samples"]))
        n_peaks = int(overrides.get("n_peaks", task.params["n_peaks"]))
        n_t_per_interval = int(overrides.get("n_t_per_interval", task.params.get("n_t_per_interval", 25)))
        large_number = float(overrides.get("LARGE_NUMBER", task.params.get("LARGE_NUMBER", 1e4)))
        seed = overrides.get("seed", task.params.get("seed", None))

        start_sampler = _sampler_from_spec(overrides.get("start_distribution", task.params["start_distribution"]))
        length_sampler = _sampler_from_spec(overrides.get("duration_distribution", task.params["duration_distribution"]))
        delay_sampler = _sampler_from_spec(overrides.get("correlation_delay_distribution", task.params["correlation_delay_distribution"]))
        ic_obj = self.build_ic(task, overrides)

        def evaluate_protocol(state: Any, rng, correlated: bool):
            events, u_sequence, nested_time_horizon = _sample_active_learning_protocol(
                rng,
                n_peaks=n_peaks,
                horizon=horizon,
                amplitude=pulse_amplitude,
                start_sampler=start_sampler,
                length_sampler=length_sampler,
                delay_sampler=delay_sampler,
                correlated=correlated,
                n_t_per_interval=n_t_per_interval,
            )

            state.reset()
            x0_list = ic_obj.get_ic(state)
            time, _, y_list, _ = state.transient_response_piecewise(
                [u_sequence],
                x0_list,
                nested_time_horizon,
                LARGE_NUMBER=large_number,
                force=True,
            )
            output = np.asarray(y_list[0][0], dtype=float)
            s_trace = _target_trace_from_protocol(u_sequence, nested_time_horizon, input_index=1)
            target_trace = s_trace if correlated else np.zeros_like(s_trace)
            error = float(np.mean((output - target_trace) ** 2))
            peak = float(np.nanmax(output))
            target_peak = float(np.nanmax(target_trace)) if target_trace.size else 0.0

            return {
                "correlated": correlated,
                "events": events,
                "u_sequence": u_sequence,
                "time": np.asarray(time, dtype=float),
                "outputs": y_list,
                "output_trace": output,
                "s_trace": s_trace,
                "target_trace": target_trace,
                "peak": peak,
                "target_peak": target_peak,
                "error": error,
            }

        def reward_fn(state: Any):
            rng = np.random.default_rng(seed)
            correlated_runs = [evaluate_protocol(state, rng, correlated=True) for _ in range(n_samples)]
            uncorrelated_runs = [evaluate_protocol(state, rng, correlated=False) for _ in range(n_samples)]

            on_error = float(np.mean([run["error"] for run in correlated_runs]))
            off_error = float(np.mean([run["error"] for run in uncorrelated_runs]))
            loss = on_weight * on_error + off_weight * off_error

            representative = correlated_runs[0]
            state.last_task_info = {
                "type": "transient response",
                "reward": loss,
                "reward type": "active_learning_custom_loss",
                "time_horizon": representative["time"],
                "trajectories": [],
                "outputs": representative["outputs"],
                "inputs": [representative["u_sequence"]],
                "active_learning_runs": correlated_runs + uncorrelated_runs,
                "correlated_peaks": np.array([run["peak"] for run in correlated_runs], dtype=float),
                "uncorrelated_peaks": np.array([run["peak"] for run in uncorrelated_runs], dtype=float),
                "correlated_target_peaks": np.array([run["target_peak"] for run in correlated_runs], dtype=float),
                "uncorrelated_target_peaks": np.array([run["target_peak"] for run in uncorrelated_runs], dtype=float),
                "on_error": on_error,
                "off_error": off_error,
                "horizon": horizon,
            }

            return loss, state.last_task_info

        return reward_fn

## Step 4: inspect the task documentation

Every task kind should explain its own parameters. This lets users inspect the task without opening its implementation.


In [ ]:
ActiveLearningTaskKind.pretty_help()

## Step 5: materialize a TaskSpec

`make_task(...)` is the public constructor. It finds `ActiveLearningTaskKind` through the registry, validates the parameters, builds the default two-input pulse vector, and exposes the resulting loss as `task.compute_reward`.

This experiment samples two families of protocols: correlated `CS/S` intervals where `O(t)` should follow `S(t)`, and uncorrelated intervals where `O(t)` should stay near zero.

In [ ]:
# collect the experiment and loss settings in one self-contained task description
task = make_task(
    template_crn=crn_template,
    library_components=library_components,
    kind="active_learning",
    species_labels=species_labels,
    params={
        "pulse_amplitude": 1.0,
        "start_distribution": lambda rng, n: rng.uniform(0.0, 20.0, size=n),
        "duration_distribution": lambda rng, n: rng.exponential(scale=1.5, size=n),
        "correlation_delay_distribution": lambda rng, n: rng.exponential(scale=0.4, size=n),
        "horizon": 25.0,
        "n_samples": 4,
        "n_peaks": 6,
        "on_weight": 1.0,
        "off_weight": 1.0,
        "n_t_per_interval": 20,
        "seed": 0,
        "ic": "zero",
        "LARGE_NUMBER": 1e4,
    },
)

print("Task kind:", task.kind)
print("Default input:", task.u_list[0].tolist())
print("Horizon:", task.params["horizon"])
print("Samples per class:", task.params["n_samples"])

## Example: score the empty template

The reward function accepts a candidate CRN and returns `(loss, info)`. Lower loss is better.

The empty template has no output, so it does well on uncorrelated protocols and poorly on correlated protocols because it cannot track `S(t)`.

In [ ]:
# clone the template because training always evaluates independent candidate networks
empty_candidate = crn_template.clone()

loss, info = task.compute_reward(empty_candidate)

print("loss:", loss)
print("correlated output peaks:", info["correlated_peaks"])
print("uncorrelated output peaks:", info["uncorrelated_peaks"])
print("O vs S error on correlated protocols:", info["on_error"])
print("O vs 0 error on uncorrelated protocols:", info["off_error"])

## Example: build a hand-made candidate

This is still not training. We add one illustrative reaction by hand to verify that the custom loss and plotting path run on a non-empty candidate.

In [ ]:
# start from the same fixed template used by the empty candidate
active_candidate = crn_template.clone()

# let simultaneous input-driven species help produce the output species
active_reaction = MassAction(
    reactant_labels=["X1", "X2"],
    product_labels=["X1", "X2", "O"],
    input_channels=[None],
    params=[0.3],
    params_controllability=[False],
)
active_candidate.add_reaction(active_reaction)

active_loss, active_info = task.compute_reward(active_candidate)

print("loss:", active_loss)
print("correlated output peaks:", active_info["correlated_peaks"])
print("uncorrelated output peaks:", active_info["uncorrelated_peaks"])
print("O vs S error on correlated protocols:", active_info["on_error"])
print("O vs 0 error on uncorrelated protocols:", active_info["off_error"])

## Example: plot exactly what the custom loss sees

The plot shows one representative correlated protocol from the sampled batch. The lower axis overlays `O(t)` with the target trace, which is `S(t)` for correlated protocols and zero for uncorrelated protocols.

In [ ]:
# read the cached trajectory produced while scoring the hand-made candidate
time = active_info["time_horizon"]
output = active_info["outputs"][0][0]
runs = active_info["active_learning_runs"]
representative = runs[0]

target = representative["target_trace"]
s_trace = representative["s_trace"]

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, height_ratios=[1, 2])

# reconstruct the two input traces from the representative event sequence
input_times = [0.0]
cs_values = [0.0]
s_values = [0.0]
for t, (cs, s) in representative["events"]:
    input_times.append(float(t))
    cs_values.append(float(cs))
    s_values.append(float(s))
input_times.append(active_info["horizon"])
cs_values.append(cs_values[-1])
s_values.append(s_values[-1])

axes[0].step(input_times, cs_values, where="post", label="CS")
axes[0].step(input_times, s_values, where="post", label="S")
axes[0].set_ylabel("input")
axes[0].legend()

axes[1].plot(time, output, color="tab:blue", linewidth=2, label="output O")
axes[1].plot(time, target, color="tab:green", linestyle=":", linewidth=2, label="target")
axes[1].set_xlabel("time")
axes[1].set_ylabel("output O")
axes[1].legend()

plt.show()

## Example: compare the two candidates

The table below exposes the correlated and uncorrelated trajectory errors. A useful active-learning candidate should make `O(t)` track `S(t)` only in correlated protocols, while keeping `O(t)` near zero for uncorrelated protocols.

In [ ]:
# evaluate each candidate and retain the measurements that explain its total loss
rows = []
for name, candidate in {
    "empty template": empty_candidate,
    "active X1+X2 -> O": active_candidate,
}.items():
    candidate_loss, candidate_info = task.compute_reward(candidate)
    rows.append(
        (
            name,
            candidate_loss,
            candidate_info["on_error"],
            candidate_info["off_error"],
            np.mean(candidate_info["correlated_peaks"]),
            np.mean(candidate_info["correlated_target_peaks"]),
            np.mean(candidate_info["uncorrelated_peaks"]),
        )
    )

for name, candidate_loss, on_error, off_error, mean_on_peak, mean_target_peak, mean_off_peak in rows:
    print(
        f"{name:22s} "
        f"loss={candidate_loss:7.3f}  "
        f"O~S error={on_error:6.3f}  "
        f"O~0 error={off_error:6.3f}  "
        f"mean O corr={mean_on_peak:6.3f}  "
        f"mean S corr={mean_target_peak:6.3f}  "
        f"mean O uncorr={mean_off_peak:6.3f}"
    )

## where this fits in a training notebook

Once a `TaskKind` is registered, the rest of RL4CRN can use it through `make_task(...)` exactly like a built-in task.

For a full training notebook, the pattern is:

1. define or import the `TaskKind`
2. build a template CRN and reaction library
3. call `make_task(..., kind="active_learning", params={...})`
4. build a `Session` from the task and a configuration
5. train, sample, and plot as in the later tutorials

For production use, move the class into a module such as `RL4CRN/utils/default_tasks/ActiveLearningTaskKind.py` and import that module before calling `make_task(...)`. Importing the module runs `@register_task_kind`.

## Minimal checklist for a new custom-loss TaskKind

- Choose a unique `kind` name.
- Decide which experiment and loss values belong in `params`.
- Implement `validate(...)` for required values and shape assumptions.
- Implement `default_u_list(...)` for default input conditions.
- Build any piecewise time grids and input protocols in `make_reward_fn(...)`.
- Return a `reward_fn(state)` that simulates the candidate and computes one scalar loss.
- Store interpretable components in `state.last_task_info` for debugging and plotting.
- Test the task on at least two simple candidates before starting training.


# Training

In [ ]:
# we define first the training configuration 
# ---- Train config ----
from RL4CRN.utils.input_interface import Configurator

cfg = Configurator.preset("fast") # build the fast Neural Network configuration (check input_interface.py for more details)

cfg.train.max_added_reactions = 5
cfg.train.epochs = 101
cfg.train.render_every = 5
cfg.train.seed = 0
cfg.render.n_best = 50
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {  # Mode of the experiment
    'style': 'logger', 
    'task': 'active_learning', 
    'format': 'image',
    'topology': True
}

cfg.describe() # print the full configuration of the experiment

Now  we define the session and the trainer

In [ ]:
import os
from datetime import datetime
from pytorch_lightning.loggers import CometLogger

from RL4CRN.utils.input_interface import make_session_and_trainer

# name the experiment and timestamp it for the comet logger
task_name = "active_learning_Task"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Expect these in your environment:
#   COMET_API_KEY   (required)
#   COMET_WORKSPACE (required)
api_key = os.environ["COMET_API_KEY"]
workspace = os.environ["COMET_WORKSPACE"]

logger = CometLogger(
    api_key=api_key,
    project=task_name,
    workspace=workspace,
    name=f"{task_name}_{timestamp}",
)

logger = logger.experiment

# creating the trainer is a one-liner, but we can also pass the logger to it
trainer = make_session_and_trainer(cfg, task, logger=logger)

Now we can train with checkpoints:

> this will print a lot of garbage... 

In [ ]:
checkpoint_path = "active_learning_task_chkpt.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)

To reload:

In [ ]:
# from RL4CRN.utils.input_interface import load_session_and_trainer

# trainer_loaded = load_session_and_trainer(checkpoint_path, device="cuda")
# trainer_loaded.inspect_best()

resimulate the current HoF

In [ ]:
hof_crns = [item.state for item in trainer.s.mult_env.hall_of_fame]

crns_new = trainer.resimulate(
    hof_crns,
    ic=("constant", 0.0),
)

trainer.inspect(crns_new[0], plot_type="transient_response_piecewise")
crns_new[0].plot_transient_response_piecewise(); plt.show()